# Getting Started
* Create a new API key at https://www.console.anthropic.com
* Create a .env file
  * Add ```ANTHROPIC_API_KEY="<your_api_key>"```
* Ensure uv is installed: https://docs.astral.sh/uv/getting-started/installation/
* Run Standalone Notebook:
  * > uv run jupyter lab
* Run Notebook in VSCode:
  * Open Jupyter Notebook and select `.venv (Python 3.12.11)` as the kernel

In [109]:
# environment setup
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from pydantic import BaseModel
client = Anthropic()
model = 'claude-haiku-4-5'
max_tokens = 1000

In [110]:
# helper functions
def add_user_message(messages, text):
  user_message = {'role': 'user', 'content': text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {'role': 'assistant', 'content': text}
  messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None, temperature=1.0):
  params = {
    'model': model,
    'max_tokens': max_tokens,
    'messages': messages,
    'temperature': temperature
  }

  if system:
    params['system'] = system

  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  message = client.messages.create(**params)

  return message.content[0].text

# Prompt Evaluation

When building prompt evaluation workflows, grading systems provide objective signals about output quality. A grader takes model output and returns some kind of measureable feedback, typically a number between 1 and 10

## Types of Graders
### Code Grader
Lets you implement any programmatic check, including:
* Checking output length
* Verifying output does/doesn't have certain words
* Syntax validation for JSON, Python, or regex
* Readability scores

### Model Grader
Feeds your original output into another API call for evaluation. This approach offers trememdous flexibility for assessing:
* Response quality
* Quality of instruction following
* Completeness
* Helpfulness
* Safety

### Human Grader
Provides the most flexibility but are time-consuming and tedious, useful for evaluating:
* General response quality
* Comprehensiveness
* Depth
* Consciseness
* Relevance

In [111]:
import json

def generate_dataset(quantity=3):
    prompt = f'''
        Generate an evaluation dataset for prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {{
                
                "task": "Description of task",
                "format": "<programming_language>"
                "solution_criteria": "Key criteria for evaluating the solution"
            }},
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code.

        Please generate {quantity} objects.
        '''
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, '```json')
    text = chat(messages, stop_sequences=['```'])

    return json.loads(text)

In [112]:
from abc import ABC
class Grader(ABC):
    def evaluate(self, test_case: dict, output: dict):
        return {}

In [113]:
'''
Before implementing any grader, you need clear evaluation criteria. For a code generation prompt, you might focus on:
* Format - should return only Python, JSON, or Regex without explanation
* Valid Syntax - Produces code should have valid syntax
* Task Following - Response should directly address the user's task with accurate code
'''
class ModelGrader(Grader):
    def evaluate(self, test_case: dict, output: dict):
        eval_prompt = f'''
            You are an expect AWS code reviewer. Your task is to evaluate the following AI-generated solution.

            Original Task:
            <task>
            {test_case['task']}
            </task>

            Solution to Evaluate:
            <solution>
            {output}
            </solution

            Criteria you should use to evaluate the solution:
            <criteria>
            {test_case['solution_criteria']}
            </criteria>

            Output Format:
            Provide your evaluation as a structured JSON object with the following fields, in this specific order:
            - "strengths": An array of 1-3 key strengths
            - "weaknesses": An array of 1-3 key areas for improvement
            - "reasoning": A concise exaplanation of your overall assessment
            - "score": A number between 1-10

            Respond with JSON. Keep your responses concise and direct.
            Example response shape:
            {{
                "strengths": string[],
                "weaknesses": string[],
                "reasoning": string,
                "score": number,
            }}
        '''

        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, '```json')
        eval_text = chat(messages, stop_sequences=['```'])

        return json.loads(eval_text)

In [114]:
import re
import ast

class CodeGrader(Grader):
    def evaluate(self, test_case: dict, output: dict):
        format = test_case['format'].lower()

        match format:
            case 'json':
                return {'score': self.validate_json(output)}
            case 'pyton':
                return {'score': self.validate_python(output)}
            case 'regex':
                return {'score': self.validate_regex(output)}
            case _:
                return {'score': 0} 
    
    def validate_json(self, text):
        try:
            json.loads(text.strip())
            return 10
        except json.JSONDecodeError:
            return 0
    
    def validate_python(self, text):
        try:
            ast.parse(text.strip())
            return 10
        except SyntaxError:
            return 0
    
    def validate_regex(self, text):
        try:
            re.compile(text.strip())
            return 10
        except re.error:
            return 0

In [115]:
def run_prompt(test_case):
    '''Merges the prompt and test case input, then returns the result'''
    prompt = f'''
    Please solve the following task:

    {test_case['task']}

    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    '''

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, '```code')
    output = chat(messages, stop_sequences=['```'])

    return output


In [116]:
def run_test_case(test_case):
    '''calls run_prompt then grades the result'''

    output = run_prompt(test_case)

    model_grade = ModelGrader().evaluate(test_case, output)
    model_score = model_grade['score']
    model_reasoning = model_grade['reasoning']

    syntax_grade = CodeGrader().evaluate(test_case, output)
    syntax_score = syntax_grade['score']

    score = (model_score + syntax_score) / 2

    return {
        'output': output,
        'test_case': test_case,
        'score': score,
        'reasoning': model_reasoning
    }

In [117]:
from statistics import mean
def run_eval(dataset):
    '''Loads the dataset and calls run_test_case with each case'''
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result['score'] for result in results])

    print(f'Average score: {average_score}')
    
    return results

In [118]:
dataset = generate_dataset()
dataset

[{'task': 'Parse an AWS S3 bucket name from an S3 URI (e.g., s3://my-bucket/path/to/file.txt) and extract only the bucket name',
  'format': 'Regex',
  'solution_criteria': 'Must correctly extract the bucket name from various S3 URI formats (with and without trailing paths), handle different bucket naming conventions, and return only the bucket identifier'},
 {'task': 'Create a Python function that takes an AWS CloudWatch log event as input and returns the log message with timestamp and log level extracted separately',
  'format': 'Python',
  'solution_criteria': 'Function must parse the CloudWatch log event structure, correctly extract the message content, timestamp, and log level (if present), and return them as a dictionary with clear key-value pairs'},
 {'task': 'Generate a JSON object representing an AWS IAM policy statement that allows read-only access to objects in a specific S3 bucket',
  'format': 'JSON',
  'solution_criteria': 'Must be valid JSON with proper IAM policy struct

In [119]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [120]:

with open('dataset.json', 'r') as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.666666666666667


In [121]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket(uri):\n    match = re.match(r's3://([a-z0-9\\-\\.]+)', uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from an S3 URI (e.g., s3://my-bucket/path/to/file.txt) and extract only the bucket name",
      "format": "Regex",
      "solution_criteria": "Must correctly extract the bucket name from various S3 URI formats (with and without trailing paths), handle different bucket naming conventions, and return only the bucket identifier"
    },
    "score": 8.0,
    "reasoning": "The solution correctly handles the basic use case of extracting bucket names from standard S3 URIs and implements defensive programming with null checking. However, it lacks comprehensive validation against AWS S3 bucket naming constraints and doesn't handle edge cases like uppercase letters or invalid naming patterns. While functional for typical URIs, it would benefit from stricter validation a